# Universal Policy Metrics & 30-Year Backtest Evaluation
This notebook provides a deep dive into the internal metrics of the Multi-Agent RL Transformer. It visualizes the latent **Asset Embeddings** learned by the model and evaluates the recommended portfolio against a 30-year historical baseline (S&P 500).

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sys
import os
from sklearn.decomposition import PCA

# Add root and backend to path
sys.path.append(os.path.abspath('.'))
sys.path.append(os.path.abspath('./backend'))

from vector_encoder import RLRecommender
from _constants import TEST_PROFILES

plt.style.use('dark_background') # Premium aesthetic

print("Initializing RL Recommender Engine...")
recommender = RLRecommender()
print(f"Engine Ready: {recommender._initialized}")
print(f"Loaded {len(recommender.dynamic_embeddings)} dynamic asset embeddings.")
print(f"Loaded {len(recommender.models)} RL Ensemble Agents.")

### 1. Asset Embedding Visualization (PCA Projection)
The transformer learns a high-dimensional embedding for each asset based on its price history, volatility, and fundamental metrics. Here, we reduce these embeddings to 2D to see how the model "groups" similar assets together.

In [ ]:
# Extract embeddings
tickers = list(recommender.dynamic_embeddings.keys())
if not tickers:
    print("ERROR: No embeddings found. Please run the data sync / ML worker first.")
else:
    embeddings = np.array([recommender.dynamic_embeddings[t] for t in tickers])

    # Reduce to 2D using Principal Component Analysis
    pca = PCA(n_components=2)
    emb_2d = pca.fit_transform(embeddings)

    # Plot the entire universe
    plt.figure(figsize=(14, 10))
    plt.scatter(emb_2d[:, 0], emb_2d[:, 1], alpha=0.3, c='#00E676', s=15, edgecolors='none')

    # Highlight and label a few recognizable major tickers
    highlight_tickers = ['AAPL', 'MSFT', 'TSLA', 'JNJ', 'JPM', 'NVDA', 'SPY', 'QQQ', 'TLT', 'GLD']
    for i, t in enumerate(tickers):
        if t in highlight_tickers:
            plt.scatter(emb_2d[i, 0], emb_2d[i, 1], c='#FF3D00', s=80, edgecolors='white', zorder=5)
            plt.annotate(t, (emb_2d[i, 0]+0.05, emb_2d[i, 1]+0.05), fontsize=12, weight='bold', color='white')

    plt.title("Asset Semantic Landscape (2D PCA of Learned Embeddings)", fontsize=16, weight='bold')
    plt.xlabel(f"Principal Component 1 ({pca.explained_variance_ratio_[0]:.1%} variance)", fontsize=12)
    plt.ylabel(f"Principal Component 2 ({pca.explained_variance_ratio_[1]:.1%} variance)", fontsize=12)
    plt.grid(alpha=0.15)
    plt.show()

### 1.1 Raw Embedding Inspection
Use the cell below to inspect the raw 8-dimensional latent vectors for any asset. You can either list specific tickers in `my_tickers` or set `show_all = True` to dump the entire universe (caution: very large).

In [ ]:
# ADJUST THESE SETTINGS
my_tickers = ['AAPL', 'MSFT', 'JNJ', 'SPY', 'BTC-USD', 'TSLA'] # Add any tickers you want here
show_all = False # Set to True to see all 6,000+ assets
limit = 50 # If show_all is True, limit the output to the first N assets

target_list = tickers if show_all else my_tickers
if show_all:
    target_list = target_list[:limit]
    print(f"Displaying first {limit} assets in the universe...\n")

print(f"{'Ticker':<10} | {'Raw Latent Vector (8-Dim)':<40}")
print("-"*65)

count = 0
for t in target_list:
    if t in recommender.dynamic_embeddings:
        vec = recommender.dynamic_embeddings[t]
        vec_str = ' '.join([f'{v:>7.3f}' for v in vec])
        print(f"{t:<10} | [{vec_str} ]")
        count += 1
    elif not show_all:
        print(f"{t:<10} | [ ERROR: Ticker not found in embedding cache ]")

if count == 0 and not show_all:
    print("\nNo valid tickers found. Please check your spelling or verify data sync.")

### 2. Multi-Goal Profile Configuration
We configure a set of goals with different horizons (e.g. Downpayment in 5 years, Retirement in 25 years). The RL model generates different weights for each "Phase" of the backtest.

In [ ]:
from vector_encoder import encode_multi_horizon

# Create a Multi-Goal request
multi_goal_answers = {
    "start_cap": 100000,
    "monthly_contrib": 1000,
    "drawdown_sensitivity": 4,
    "volatility_sensitivity": 4,
    "goals": [
        {"name": "House Downpayment", "amount": 200000, "years": 5},
        {"name": "Retirement", "amount": 2000000, "years": 25}
    ]
}

print("Requesting Multi-Horizon weights from the Agent Ensemble...")
res = encode_multi_horizon(multi_goal_answers)

# FIX: encode_multi_horizon returns a flat dictionary, not nested under 'portfolio'
segments = res.get('segments', [])

if not segments:
    print("\nWARNING: No segments were generated. Diagnostics:")
    print(f"  - Recommender Models Loaded: {len(recommender.models)}")
    print(f"  - Input Goals: {len(multi_goal_answers.get('goals', []))}")
    print(f"  - Full API Response Keys: {list(res.keys())}")
    if 'error' in res:
        print(f"  - ERROR MESSAGE: {res['error']}")
else:
    print(f"Successfully generated {len(segments)} segments.")
    for i, seg in enumerate(segments):
        print(f"\nPHASE {i+1}: {seg['goal_name']} (Years {seg['horizon_years'][0]} - {seg['horizon_years'][1]})")
        w = seg.get('weights', {})
        if not w:
            print("  -> No assets selected for this phase (weights are empty).")
        else:
            for t, val in sorted(w.items(), key=lambda x: x[1], reverse=True)[:5]:
                print(f"  {t:<8}: {val:>6.2%}")

### 3. Multi-Phase 30-Year POT Backtest
This simulation switches the portfolio weights as it passes through the "Phase" boundaries (Target Date Fund logic) over a 30-year historical window.

In [ ]:
import matplotlib.ticker as ticker

if not segments:
    print("Skipping backtest because no segments were generated.")
else:
    returns = recommender.daily_returns
    years_to_test = 30
    days_to_test = years_to_test * 252

    if len(returns) < days_to_test:
        days_to_test = len(returns)

    back_returns = returns.iloc[-days_to_test:]
    dates = back_returns.index

    # Get all unique assets across all phases
    all_assets = set()
    for seg in segments:
        all_assets.update(seg['weights'].keys())
    all_assets = [t for t in all_assets if t in back_returns.columns]
    if not all_assets: all_assets = ['SPY']

    port_returns_matrix = back_returns[all_assets]

    # Build dynamic target_weights per day based on historical Phase
    target_df = pd.DataFrame(0.0, index=dates, columns=all_assets)
    for i, dt in enumerate(dates):
        years_ago = (dates[-1] - dt).days / 365.25
        # Match to segment
        chosen_seg = segments[-1]
        for seg in segments:
            if seg['horizon_years'][0] <= years_ago < seg['horizon_years'][1]:
                chosen_seg = seg
                break
        for t, w in chosen_seg['weights'].items():
            if t in all_assets:
                target_df.loc[dt, t] = w

    # Hindsight Bias Protection / Proxying
    active_mask = port_returns_matrix.notna()
    daily_weights = active_mask * target_df
    missing_weight = 1.0 - daily_weights.sum(axis=1)

    spy_returns = back_returns['^GSPC'] if '^GSPC' in back_returns.columns else back_returns['SPY'] if 'SPY' in back_returns.columns else back_returns.mean(axis=1)
    spy_returns = spy_returns.fillna(0.0)

    port_daily_rets = (port_returns_matrix.fillna(0.0) * daily_weights).sum(axis=1) + (spy_returns * missing_weight)

    # Cumulative
    initial_capital = multi_goal_answers.get('start_cap', 100000)
    port_cum = (1 + port_daily_rets).cumprod() * initial_capital
    spy_cum = (1 + spy_returns).cumprod() * initial_capital

    plt.figure(figsize=(16, 8))
    plt.plot(dates, port_cum, label='RL Multi-Goal Portfolio (Phase-Shifted)', color='#00E676', linewidth=2)
    plt.plot(dates, spy_cum, label='S&P 500 Baseline', color='#B0BEC5', linewidth=1.5, linestyle='--')

    # Mark Phase Boundaries in the Plot
    for i, seg in enumerate(segments[:-1]):
        boundary_years = seg['horizon_years'][1]
        boundary_date = dates[-1] - pd.Timedelta(days=boundary_years * 365.25)
        plt.axvline(boundary_date, color='white', alpha=0.2, linestyle=':')
        plt.text(boundary_date, plt.ylim()[1], f" Phase {i+1} Ends", rotation=90, verticalalignment='top', alpha=0.5)

    plt.title("30-Year Multi-Goal Backtest: RL Glide Path vs Benchmark", fontsize=16, weight='bold')
    plt.ylabel("Portfolio Value ($)", fontsize=12)
    plt.yscale('log')
    plt.grid(alpha=0.15)
    plt.legend(loc='upper left')
    plt.gca().yaxis.set_major_formatter(ticker.FuncFormatter(lambda y, pos: f'${y:,.0f}'))
    plt.show()

    print(f"Final Multi-Goal Value: ${port_cum.iloc[-1]:,.0f}")
    print(f"Final S&P 500 Value:    ${spy_cum.iloc[-1]:,.0f}")